In [ ]:
"""Hostel Borrowing Club - an OOP project in Python.

Run the interactive app :  python hostel_club.py
Run a scripted demo     :  python hostel_club.py --demo
"""
import sys
from abc import ABC, abstractmethod
from datetime import date, timedelta


# ----------------------------------------------------------------------
# Custom exceptions
# ----------------------------------------------------------------------
class ClubError(Exception):
    """Base class for all club errors."""


class ItemUnavailableError(ClubError):
    pass


class LimitReachedError(ClubError):
    pass


class AuthError(ClubError):
    pass


# ----------------------------------------------------------------------
# INTERFACE / ABSTRACT CLASS + ABSTRACTION
# Borrowable says WHAT every lendable thing must offer, not HOW.
# ----------------------------------------------------------------------
class Borrowable(ABC):
    @property
    @abstractmethod
    def loan_days(self) -> int: ...

    @property
    @abstractmethod
    def fee_per_day(self) -> int: ...

    @abstractmethod
    def describe(self) -> str: ...


# ----------------------------------------------------------------------
# CLASSES, CONSTRUCTORS, ENCAPSULATION, INHERITANCE, POLYMORPHISM
# ----------------------------------------------------------------------
class Item(Borrowable):
    """Abstract-ish parent of all items. Implements the shared behaviour."""

    _next_id = 1          # class attribute shared by all objects
    category = "General"

    def __init__(self, name: str):          # constructor
        self._id = Item._next_id
        Item._next_id += 1
        self._name = name                    # protected
        self.__available = True              # private (name-mangled)

    @property
    def id(self):
        return self._id

    @property
    def name(self):
        return self._name

    @property
    def available(self):                     # read-only from outside
        return self.__available

    def check_out(self):                     # controlled way to change state
        if not self.__available:
            raise ItemUnavailableError(f"'{self._name}' is already borrowed.")
        self.__available = False

    def check_in(self):
        self.__available = True

    def describe(self):
        state = "available" if self.__available else "borrowed"
        return (f"#{self._id:<2} {self._name:<18} {self.category:<11} "
                f"{self.loan_days}d  UGX {self.fee_per_day}/day late  [{state}]")

    @classmethod
    def from_dict(cls, data: dict):          # alternative constructor
        kinds = {"electronic": Electronic, "utensil": Utensil, "book": Book}
        return kinds[data["kind"]](data["name"])

    def __repr__(self):
        return f"{type(self).__name__}({self._name!r})"


class Electronic(Item):
    category = "Electronic"
    loan_days = 2            # each subclass fills in the abstract members
    fee_per_day = 2000

    def describe(self):      # polymorphism: overrides parent behaviour
        return super().describe() + "  (handle with care)"


class Utensil(Item):
    category = "Utensil"
    loan_days = 3
    fee_per_day = 500


class Book(Item):
    category = "Book"
    loan_days = 7
    fee_per_day = 300


class Person:
    def __init__(self, name: str, phone: str):
        self._name = name
        self._phone = phone

    @property
    def name(self):
        return self._name

    @property
    def phone(self):
        return self._phone

    def describe(self):
        return f"{self._name} ({self._phone})"


class Member(Person):
    MAX_LOANS = 3
    _next_id = 1

    def __init__(self, name, phone, room, pin="0000"):
        super().__init__(name, phone)        # reuse the parent constructor
        self._id = Member._next_id
        Member._next_id += 1
        self.room = room
        self.__pin = pin                     # private: never exposed
        self._loans = []                     # association: Member --- Loan

    @property
    def id(self):
        return self._id

    def verify_pin(self, pin) -> bool:
        return pin == self.__pin

    def active_loans(self):
        return [l for l in self._loans if l.returned_on is None]

    def can_borrow(self) -> bool:
        return len(self.active_loans()) < self.MAX_LOANS

    def describe(self):                      # polymorphism
        return (f"#{self._id:<2} {self._name:<14} room {self.room:<5} "
                f"{self._phone}  loans: {len(self.active_loans())}/{self.MAX_LOANS}")

class Admin(Member):
    """Inheritance: an Admin IS a Member with extra powers."""

    MAX_LOANS = 5                            # overrides class attribute

    def issue_loan(self, member, item, today, notifier):
        """notifier is only USED here (dependency), never stored."""
        if not member.can_borrow():
            raise LimitReachedError(f"{member.name} reached the limit of {member.MAX_LOANS} items.")
        item.check_out()
        loan = Loan(member, item, today)
        member._loans.append(loan)
        notifier.send(member, f"You borrowed {item.name}. Return by {loan.due_on}.")
        return loan

    def receive_return(self, loan, today, notifier):
        loan.close(today)
        msg = f"{loan.item.name} returned."
        if loan.penalty:
            msg += f" Late penalty: {loan.penalty}."
        notifier.send(loan.member, msg)
        return loan


class Penalty:
    def __init__(self, days_late: int, amount: int):
        self.days_late = days_late
        self.amount = amount

    def __str__(self):
        return f"UGX {self.amount} ({self.days_late} day(s) late)"


class Loan:
    """Association class linking one Member to one Item.
    COMPOSITION: the Penalty is created inside the Loan and dies with it."""

    _next_id = 1

    def __init__(self, member, item, borrowed_on):
        self.id = Loan._next_id
        Loan._next_id += 1
        self.member = member
        self.item = item
        self.borrowed_on = borrowed_on
        self.due_on = borrowed_on + timedelta(days=item.loan_days)  # polymorphic
        self.returned_on = None
        self.penalty = None

    def is_overdue(self, today):
        return self.returned_on is None and today > self.due_on

    def close(self, today):
        self.returned_on = today
        self.item.check_in()
        late = (today - self.due_on).days
        if late > 0:
            self.penalty = Penalty(late, late * self.item.fee_per_day)

    def describe(self):
        status = "returned" if self.returned_on else "OUT"
        pen = f"  penalty {self.penalty}" if self.penalty else ""
        return (f"Loan {self.id}: {self.member.name} -> {self.item.name} "
                f"due {self.due_on} [{status}]{pen}")